In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import calpred
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import List
import statsmodels.api as sm
from scipy import stats
import pickle
from tqdm import tqdm
import calpgs
import copy
import warnings

import pandas as pd
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
import matplotlib as mpl

mpl.rcParams['font.family'] = "Nimbus Sans"
mpl.rcParams['lines.linewidth'] = 1.5
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['font.size'] = 16

# simulate Genetic Value (GV)

In [ ]:
#############################################
# Assume simulated genotype as binary bfiles is named "geno_82k_hm3_chr1"
# Simulation process follows https://github.com/xuchang0201/PredInterval/blob/main/Manuscript/Simulations/Main_Simulations_Quantitative.R, commit 276822c
#############################################
# This part can be skipped. DataFrames with trained PGS across 5 CV are also available.
# See df_woContext.zip
# If to skip [simulate GV, simulate phenotype, GWAS and PGS], go directly to CalPred and PredInterval
#############################################

In [ ]:
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:

        for seed in tqdm(range(30)):
        
            snplist = pd.read_csv("geno_82k_hm3_chr1.bim",sep="\t",header=None)
            n_snp = len(snplist)
        

        
            n_causal_snps=int(causal_percent*n_snp)
        
            causal_snp = snplist.sample(n_causal_snps, replace=False,random_state = seed).rename(columns={1:"ID"})
            
            np.random.seed(seed)
            causal_snp["beta"] = np.random.normal(0,np.sqrt(h2/n_causal_snps),n_causal_snps)
            freq = pd.read_csv("geno_82k_hm3_chr1.afreq",sep="\t")
            causal_snp = causal_snp.merge(freq[["ID","ALT_FREQS"]],on="ID",how="inner")
            causal_snp["beta_freqAdj"] = causal_snp.beta/np.sqrt(2*causal_snp.ALT_FREQS*(1-causal_snp.ALT_FREQS))
            
            causal_snp.to_csv(f"sim.h2{h2}.causal{causal_percent}.seed{seed+1}.tsv",sep="\t",index=None)
        


In [ ]:
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for i in range(1,31):
            !plink \
                --bfile geno_82k_hm3_chr1 \
                --score sim.h2{h2}.causal{causal_percent}.seed{i}.tsv 2 6 9 header cols=+scoresums \
                --threads 8 \
                --out sim_gv.h2{h2}.causal{causal_percent}.seed{i}

        

# simulate phenotype

In [ ]:
#############################################
# Simulation process follows https://github.com/xuchang0201/PredInterval/blob/main/Manuscript/Simulations/Main_Simulations_Quantitative.R, commit 276822c
#############################################
# This part can be skipped. DataFrames with trained PGS across 5 CV are also available.
# See df_woContext.zip
# If to skip [simulate GV, simulate phenotype, GWAS and PGS], go directly to CalPred and PredInterval
#############################################

In [ ]:

for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
    
        fam = pd.read_csv("geno_82k_hm3_chr1.fam",header=None,sep="\t")
        np.random.seed(42)
        fam = fam[np.arange(2)]
        for i in tqdm(range(30)):
            gv = pd.read_csv(f"sim_gv.h2{h2}.causal{causal_percent}.seed{i+1}.sscore",sep="\t")
            fam[3] = gv.SCORE1_SUM
            
            fam[4] = np.random.normal(
                loc=fam[3],
                scale=np.sqrt(1-h2),
            )

            fam[4] = (fam[4]-fam[4].mean())/fam[4].std()
            df_trait = copy.copy(fam)
            df_trait.columns=["FID","IID","GV","y"]

            df_trait.sample(frac=1,random_state=i).iloc[:60000].to_csv(f"sim_df.h2{h2}.causal{causal_percent}.seed{i+1}.tsv",sep="\t",index=None)
    


# GWAS

In [ ]:
#############################################
# This part can be skipped. DataFrames with trained PGS across 5 CV are also available.
# See df_woContext.zip
# If to skip [simulate GV, simulate phenotype, GWAS and PGS], go directly to CalPred and PredInterval
#############################################

In [ ]:

for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for i in tqdm(range(30)):
            df = pd.read_csv(f"sim_df.h2{h2}.causal{causal_percent}.seed{i+1}.tsv",sep="\t")
            df = df.iloc[:50000]
            pheno = df[["FID","IID","y"]]
            
            for j in range(5):
                index = df.iloc[j*10000:(j+1)*10000].IID
                pdpheno = pheno[~pheno.IID.isin(index)]
                pdpheno.to_csv(f"pheno.h2{h2}.causal{causal_percent}.seed{i}.cv{j}.tsv",sep="\t",index=None)

        
    

In [ ]:
#############################################
# Consider parallel in your environment
#############################################
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for i in tqdm(range(30)):
            for j in range(5):
                !plink \
                    --bfile geno_82k_hm3_chr1 \
                    --pheno pheno.h2{h2}.causal{causal_percent}.seed{i}.cv{j}.tsv \
                    --pheno-col-nums 3 \
                    --glm allow-no-covars \
                    --no-input-missing-phenotype \
                    --threads 8 \
                    --out gwas.h2{h2}.causal{causal_percent}.seed{i}.cv{j}

                    

# PGS

In [ ]:
#############################################
# This part can be skipped. DataFrames with trained PGS across 5 CV are also available.
# See df_woContext.zip
# If to skip [simulate GV, simulate phenotype, GWAS and PGS], go directly to CalPred and PredInterval
#############################################

In [ ]:
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for seed in tqdm(range(30)):
            for cv in range(5):
                sst = pd.read_csv(f"gwas.h2{h2}.causal{causal_percent}.seed{seed}.cv{cv}.y.glm.linear",sep="\t").drop(columns=["A1"])
                sst.rename(columns={"ID":"SNP","ALT":"A1","REF":"A2"},inplace=True)
                sst[["SNP","A1","A2","BETA","SE"]].to_csv(f"gwas.h2{h2}.causal{causal_percent}.seed{seed}.cv{cv}.sst",sep="\t",index=None)


In [ ]:
#############################################
# Consider parallel in your environment
#############################################
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for seed in tqdm(range(30)):
            for cv in range(5):
                !python PRScs.py \
                    --ref_dir=ldblk_1kg_eur \
                    --bim_prefix=geno_82k_hm3_chr1 \
                    --sst_file=gwas.h2{h2}.causal{causal_percent}.seed{seed}.cv{cv}.sst \
                    --n_gwas=40000 \
                    --chrom=1 \
                    --out_dir=pgs.h2{h2}.causal{causal_percent}.seed{seed}.cv{cv}
                


In [ ]:
#############################################
# Consider parallel in your environment
#############################################
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for seed in tqdm(range(30)):
            for cv in range(5):
                !plink \
                    --bfile geno_82k_hm3_chr1 \
                    --score pgs.h2{h2}.causal{causal_percent}.seed{seed}.cv{cv}_pst_eff_a1_b0.5_phiauto_chr1.txt 2 4 6 cols=+scoresums \
                    --threads 8 \
                    --out score.h2{h2}.causal{causal_percent}.seed{seed}.cv{cv}




In [ ]:
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for i in tqdm(range(30)):
            df = pd.read_csv(f"sim_df.h2{h2}.causal{causal_percent}.seed{i+1}.tsv",sep="\t")
            for cv in range(5):
                pgs = pd.read_csv(f"score.h2{h2}.causal{causal_percent}.seed{i}.cv{cv}.sscore",sep="\t")
                df = df.merge(pgs[["IID","SCORE1_SUM"]].rename(columns={"SCORE1_SUM":f"PGS_{cv}"}),on="IID",how="left")
                
            df.to_csv(f"sim_df_PGS.h2{h2}.causal{causal_percent}.seed{i}.tsv",sep="\t",index=None)


# CalPred 

In [ ]:
#############################################
# Follows https://github.com/KangchengHou/calpred-manuscript/blob/main/experiments/simulation/simulate.ipynb
#############################################
# This part can be skipped. Results are also available, see result_woContext.zip
# If to skip training, go directly to plot_wocontext.ipynb
#############################################

In [ ]:
def evaluate(alpha,N_SIM=10,correct=True,causal_percent=1,h2=0.5):

    dict_stats_sum = dict()

    adjust="none"
    adjust_cols = []

    dict_df_coverage = dict()
    dict_df_r2 = dict()
    dict_df_length = dict()
    #dict_df_params = dict()

    df_coverage = []
    df_r2 = []
    df_length = []
    df_params = []
    for seed in tqdm(range(N_SIM)):
        df = pd.read_csv(f"sim_df_PGS.h2{h2}.causal{causal_percent}.seed{seed}.tsv",sep="\t")

        if correct:
            df_cal = df.iloc[:10000]
        else:
            df_cal = df.iloc[10000:50000]
        df_test = df.iloc[50000:]

    


        model = calpred.fit(y=df_cal["y"],x=df_cal["PGS_0"],z=df_cal[adjust_cols])
        mean,std = calpred.predict(x=df_test["PGS_0"],z=df_test[adjust_cols],model_fit=model)
        df_test["mean"] = mean
        df_test["std"] = std
        df_test["lower"] = mean-stats.norm.ppf(0.5 + alpha/2)*std
        df_test["upper"] = mean+stats.norm.ppf(0.5 + alpha/2)*std

        if correct:
            df_test[["IID","y","mean","std","lower","upper"]].to_csv(f"calPredCorrect.alpha{alpha}.h2{h2}.causal{causal_percent}.seed{seed}.pred.tsv",sep="\t",index=None)
        else:
            df_test[["IID","y","mean","std","lower","upper"]].to_csv(f"calPred.alpha{alpha}.h2{h2}.causal{causal_percent}.seed{seed}.pred.tsv",sep="\t",index=None)


        tmp_cov = {}
        tmp_r2 = {}
        tmp_length = {}
        for group_col in [None]:
            if group_col == None:
                tmp_cov["marginal"]=df_test["y"].between(df_test.lower,df_test.upper).sum()/len(df_test)
                tmp_r2["marginal"]=stats.pearsonr(df_test["PGS_0"],df_test.y).statistic**2
                tmp_length["marginal"] = df_test["std"].mean()
                continue
            
        tmp_cov = pd.Series(tmp_cov)
        tmp_r2= pd.Series(tmp_r2)
        tmp_length = pd.Series(tmp_length)
        
        df_coverage.append(tmp_cov)
        df_r2.append(tmp_r2)
        df_length.append(tmp_length)
        n_calibrate = 10000
        dict_df_coverage[n_calibrate] = pd.concat(df_coverage, axis=1).T
        dict_df_r2[n_calibrate] = pd.concat(df_r2, axis=1).T
        dict_df_length[n_calibrate] = pd.concat(df_length, axis=1).T
        #dict_df_params[n_calibrate] = pd.concat(df_params, axis=1).T

        df_stats = {
            "n": [],
            "seed": [],
            "col": [],
            "coverage": [],
            "r2": [],
            "length": [],
        }

        for n in dict_df_coverage:
            for col in dict_df_coverage[n].columns:
                covs = dict_df_coverage[n][col].values
                df_stats["n"].extend([n] * len(covs))
                df_stats["seed"].extend(np.arange(len(covs)))
                df_stats["col"].extend([col] * len(covs))
                df_stats["coverage"].extend(covs)
                df_stats["r2"].extend(dict_df_r2[n][col])
                df_stats["length"].extend(dict_df_length[n][col])

        
        dict_stats_sum[adjust] = pd.DataFrame(df_stats)

    df_stats = []
    for adjust in dict_stats_sum:
        df_tmp = dict_stats_sum[adjust]
        df_tmp.insert(0, "adjust", adjust)
        df_stats.append(df_tmp)
    df_stats = pd.concat(df_stats)


    if correct:
        df_stats.to_csv(f"calPredCorrect.alpha{alpha}.h2{h2}.causal{causal_percent}.stats.tsv", sep="\t", index=False)
    else:
        df_stats.to_csv(f"calPred.alpha{alpha}.h2{h2}.causal{causal_percent}.stats.tsv", sep="\t", index=False)



    
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        for correct in [True,False]:

            evaluate(0.95,N_SIM=30,correct=correct,causal_percent=causal_percent,h2=h2)


# PredInterval

In [ ]:
#############################################
# Follows https://github.com/xuchang0201/PredInterval/blob/main/src/PredInterval.R
#############################################
# This part can be skipped. Results are also available, see result_woContext.zip
# If to skip training, go directly to plot_wocontext.ipynb
#############################################

In [ ]:
#############################################
# Consider parallel in your environment
#############################################

def PredInterval_fit_pred(df_train,df_test,scheme,alpha):

    ricv = []

    print("start building RiCV")


    for i in range(5):
    
        current = df_train.iloc[i*10000:(i+1)*10000]
        ricv.append(abs(current[f"PGS_{i}"]-current["y"]).values) 

    
    upper = []
    lower = []
    means = []

    print("start calculating test residuals")

    
    
    for i in range(5):
        resid = ricv[i]
        mean = df_test[f"PGS_{i}"]
        means.append(mean)
        for r in resid:
            upper.append(mean+r)
            lower.append(mean-r)
    
    upper = pd.concat(upper,axis=1)
    upper = np.quantile(upper,alpha,axis=1)
    lower = pd.concat(lower,axis=1)
    lower = np.quantile(lower,1-alpha,axis=1)
    means = pd.concat(means,axis=1).mean(axis=1)
    
    std = (np.array(upper)-np.array(lower))/stats.norm.ppf(0.5 + alpha/2)/2
    

    return means, std, upper ,lower

def evaluate_PredInterval_main(df_train,df_test,scheme,seed,alpha,h2):

    print("start fitting predint")
    mean,std,upper,lower = PredInterval_fit_pred(df_train,df_test,scheme,alpha)
    print("Done")
    df_test["upper"] = upper
    df_test["lower"] = lower
    df_test["std"] = std
    df_test["mean"] = mean
    df_test[["IID","y","mean","std","lower","upper"]].to_csv(f"{scheme}.alpha{alpha}.h2{h2}.causal{causal_percent}.seed{seed}.pred.tsv",sep="\t",index=None)

    tmp_cov = {}
    tmp_r2 = {}
    tmp_length = {}
    for group_col in [None]:
        if group_col == None:
            tmp_cov["marginal"]=df_test["y"].between(df_test.lower,df_test.upper).sum()/len(df_test)
            tmp_r2["marginal"]=stats.pearsonr(df_test["mean"],df_test.y).statistic**2
            tmp_length["marginal"] = df_test["std"].mean()
            continue
        
    tmp_cov = pd.Series(tmp_cov)
    tmp_r2= pd.Series(tmp_r2)
    tmp_length = pd.Series(tmp_length)
    
    return (
        pd.Series(tmp_cov),
        pd.Series(tmp_r2),
        pd.Series(tmp_length),
    )


def evaluate_PredInterval_wrapper(alpha,sim,scheme="PredInterval",causal_percent=None,h2=0.5):
    

    
    dict_stats_sum = dict()
    dict_params_sum = dict()
    
    dict_df_coverage = dict()
    dict_df_r2 = dict()
    dict_df_length = dict()
    dict_df_params = dict()
    
    df_coverage = []
    df_r2 = []
    df_length = []
    seed = int(sim)-1
    df = pd.read_csv(f"sim_df_PGS.h2{h2}.causal{causal_percent}.seed{seed}.tsv",sep="\t")
    df_train = df.iloc[:50000] 
    df_test = df.iloc[50000:]

    tmp_cov, tmp_r2, tmp_length = evaluate_PredInterval_main(
        df_train, df_test,scheme,seed,alpha,h2
    )
    df_coverage.append(tmp_cov)
    df_r2.append(tmp_r2)
    df_length.append(tmp_length)
    n_calibrate = 10000
    dict_df_coverage[n_calibrate] = pd.concat(df_coverage, axis=1).T
    dict_df_r2[n_calibrate] = pd.concat(df_r2, axis=1).T
    dict_df_length[n_calibrate] = pd.concat(df_length, axis=1).T
    df_stats = {
        "n": [],
        "seed": [],
        "col": [],
        "coverage": [],
        "r2": [],
        "length": [],
    }
    
    # summarize coverage / R2
    for n in dict_df_coverage:
        for col in dict_df_coverage[n].columns:
            covs = dict_df_coverage[n][col].values
            df_stats["n"].extend([n] * len(covs))
            df_stats["seed"].extend(np.arange(len(covs)))
            df_stats["col"].extend([col] * len(covs))
            df_stats["coverage"].extend(covs)
            df_stats["r2"].extend(dict_df_r2[n][col])
            df_stats["length"].extend(dict_df_length[n][col])
    
    
    
    dict_stats_sum[scheme] = pd.DataFrame(df_stats)
    
    # format into long table
    df_stats = []
    for adjust in dict_stats_sum:
        df_tmp = dict_stats_sum[adjust]
        df_tmp.insert(0, "adjust", adjust)
        df_stats.append(df_tmp)
    df_stats = pd.concat(df_stats)


    
    df_stats.to_csv(f"{scheme}.alpha{alpha}.h2{h2}.causal{causal_percent}.{seed}.stats.tsv", sep="\t", index=False)



#############################################
# Consider parallel in your environment
#############################################

alpha=0.95
for sim in range(1,31):
    for h2 in [0.2,0.5,0.8]:
        for scheme in ["PredInterval"]:
            for causal_percent in [0.001,0.01,0.1,1]:
                evaluate_PredInterval_wrapper(alpha,sim,scheme,causal_percent,h2)


In [ ]:
scheme = "PredInterval"
alpha=0.95
for h2 in [0.2,0.5,0.8]:
    for causal_percent in [0.001,0.01,0.1,1]:
        df = []
        for i in range(30):
            df.append(pd.read_csv(f"{scheme}.alpha{alpha}.h2{h2}.causal{causal_percent}.{i}.stats.tsv",sep="\t"))
            
        pd.concat(df).to_csv(f"{scheme}.alpha{alpha}.h2{h2}.causal{causal_percent}.stats.tsv",sep="\t",index=None)
